# SynAgent: generate and rank synthesis routes

This notebook does two things with a molecule you want to make:

1. Generate: you give SynAgent a target molecule and your constraints, and it proposes several synthesis routes. Every step comes from its libraries of reaction templates and purchasable building blocks, and SynAgent checks that each reaction gives the product it claims.
2. Rank: SynAgent orders those routes by how feasible they are to carry out, taking your own priorities into account.

SynAgent runs on Google's Gemini model in this notebook. To use it, you need a free Google account.

## One-time setup

You only need to do these six steps once. If someone has already set SynAgent up on your computer, skip to "Opening the notebook" below.

There are two ways to install SynAgent. The steps below use uv, which also installs Python for you. If you'd rather not install uv, or can't, follow the "Without uv" instructions in steps 3 and 4 instead.

### 1. Open a terminal

A terminal is a window where you type commands. Paste each command below into it and press Enter.

- macOS: press `⌘ Space`, type `Terminal`, and press Enter.
- Windows: open the Start menu, type `PowerShell`, and press Enter.
- Linux: open your Terminal application.

### 2. Install git

git downloads SynAgent and one of the packages it depends on.

- macOS: type `git --version` and press Enter. If git isn't installed, macOS offers to install it. Click **Install** and wait for it to finish.
- Windows: download the installer from https://git-scm.com/downloads/win and run it with the default options.
- Linux: install it with your package manager, for example `sudo apt install git`.

### 3. Install uv

uv installs Python and the packages SynAgent needs into the project's own folder, so it doesn't change anything else on your computer.

macOS or Linux:

```sh
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Windows (PowerShell):

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Then close the terminal and open a new one so it can find `uv`.

#### Without uv: install Python

You need Python 3.13 or newer. Download it from https://www.python.org/downloads/ and run the installer. On Windows, tick **Add python.exe to PATH** on the installer's first screen.

Close the terminal, open a new one, and check the version. It should print 3.13 or higher.

- macOS or Linux: `python3 --version`
- Windows: `py --version`

### 4. Download and install SynAgent

```sh
git clone https://github.com/lukasmki/SynAgent.git
cd SynAgent
uv sync --extra jupyter
```

The first command creates a `SynAgent` folder, the second moves into it, and the third installs everything. The third one takes a few minutes the first time and prints a lot of text, which is normal.

#### Without uv

These commands create a private Python environment in a `.venv` folder inside `SynAgent`, switch the terminal to it, and install SynAgent and Jupyter there. The first `pip install` gets the version of FPSim2 that SynAgent needs, so run the two `pip install` lines in this order.

macOS or Linux:

```sh
git clone https://github.com/lukasmki/SynAgent.git
cd SynAgent
python3 -m venv .venv
source .venv/bin/activate
pip install "fpsim2 @ git+https://github.com/lukasmki/FPSim2.git"
pip install ".[jupyter]"
```

Windows (PowerShell):

```powershell
git clone https://github.com/lukasmki/SynAgent.git
cd SynAgent
py -m venv .venv
.venv\Scripts\Activate.ps1
pip install "fpsim2 @ git+https://github.com/lukasmki/FPSim2.git"
pip install ".[jupyter]"
```

Once the environment is active, the terminal prompt starts with `(.venv)`. If Windows says that running scripts is disabled, see "If something goes wrong" at the bottom of the notebook.

### 5. Get a Gemini API key

An API key works like a password that lets SynAgent use Gemini on your behalf.

1. Go to https://aistudio.google.com/api-keys and sign in with a Google account.
2. Click **Create API key**. If it asks for a project, accept the default.
3. Copy the key. It's a long string that starts with `AIza`.

Don't share the key or paste it into this notebook. Google's free tier has usage limits, and the same page shows how much you've used.

### 6. Save the key in a `.env` file

SynAgent reads the key from a file named `.env` in the `SynAgent` folder. In the terminal, still inside that folder, run the command for your system. Replace `paste-your-key-here` with the key you copied, and keep the single quotes around it.

macOS or Linux:

```sh
echo 'GOOGLE_API_KEY=paste-your-key-here' > .env
```

Windows (PowerShell):

```powershell
Set-Content -Path .env -Value 'GOOGLE_API_KEY=paste-your-key-here'
```

The file now holds one line, `GOOGLE_API_KEY=AIza...`. Finder and File Explorer hide files whose names start with a dot, so you may not see it. The project's git settings ignore the file, so it stays on your computer and is never uploaded to GitHub.

## Opening the notebook

Do this each time you want to use the notebook. Open a terminal and run:

```sh
cd SynAgent
uv run --extra jupyter jupyter lab notebook-examples/generate-and-rank.ipynb
```

If you installed without uv, switch to the project's environment first, then start Jupyter.

macOS or Linux:

```sh
cd SynAgent
source .venv/bin/activate
jupyter lab notebook-examples/generate-and-rank.ipynb
```

Windows (PowerShell):

```powershell
cd SynAgent
.venv\Scripts\Activate.ps1
jupyter lab notebook-examples/generate-and-rank.ipynb
```

The notebook opens in your web browser. Keep the terminal window open while you work, because closing it stops the notebook.

### Running the cells

- To run a code cell, click it and press **Shift + Enter**. Run the cells in order from top to bottom.
- A running cell shows `[*]` to its left. The `*` turns into a number when the cell finishes.
- Generating routes takes a few minutes, and so does ranking them. SynAgent shows its progress while it works.
- If you change a setting, run that cell again, then the cells below it.
- To start over, choose **Kernel → Restart Kernel** from the menu and run everything again from Step 1.

## Step 1: Check your setup

Click the cell below and press **Shift + Enter**. It should say that it found your API key. If it doesn't, see [If something goes wrong](#If-something-goes-wrong) at the bottom of the notebook.

In [1]:
import os

import synagent as sa
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if os.environ.get("GOOGLE_API_KEY"):
    print("✅ Found your Gemini API key. You're ready to go.")
else:
    print(
        "❌ No Gemini API key found.\n\n"
        "Check steps 5 and 6 of the setup above: the SynAgent folder needs a file\n"
        "named .env containing one line, GOOGLE_API_KEY=your-key. Once you've fixed\n"
        "it, use Kernel → Restart Kernel in the menu, then run this cell again."
    )

✅ Found your Gemini API key. You're ready to go.


## Step 2: Start SynAgent

To try a different Gemini model, change `MODEL`.

In [2]:
MODEL = "google:gemini-flash-latest"  # "anthropic:claude-sonnet-5-5"

agent = sa.SynAgent(spec="nogpu", model=MODEL)
agent

SynAgent(spec='nogpu', model='google:gemini-flash-latest')
  capabilities: AnalogueSearch, SynthesisValidation, Chemspace

## Step 3: Generate routes

### Edit this

- `TARGET` is the molecule you want to make, written as a SMILES string. To find one, look the molecule up on [PubChem](https://pubchem.ncbi.nlm.nih.gov/) and copy the SMILES under "Names and Identifiers". You can also draw it in ChemDraw or [Ketcher](https://lifescience.opensource.epam.com/KetcherDemoSA/index.html) and copy it as SMILES. Keep the quotes around it.
- `NUMBER_OF_ROUTES` is how many different routes to propose. Start with three; asking for more takes longer.
- `CONSTRAINTS` lists anything the routes must respect, in plain English, one per line. Keep the triple quotes (`"""`) at the start and end. If you have no constraints, write `None` between them.

Some other constraints you could use: *at most 4 steps*, *avoid protecting groups*, *avoid cryogenic conditions*, *start from commercially available anilines*, *keep the piperazine intact*.

In [3]:
TARGET = "O=C(C1=CC=CC(C(F)(F)F)=C1)NC2=CC(C3=CC4=C(C(C=CC=C5)=C5C=C4)N3)=CC=N2"

NUMBER_OF_ROUTES = 3

CONSTRAINTS = """
At most 3 steps.
Can contain templates that are not in the database as long as the step is valid.
"""

Run the cell below to generate the routes. It takes a few minutes.

While it works, SynAgent prints each tool it calls in a yellow box, such as a structure check or a building block search, and what the tool returned in a green box. You can ignore these. At the end, each route is drawn one reaction per step, followed by its building blocks.

In [4]:
from IPython.display import Markdown, display

chat = agent.chat()

routes = await chat.ask(
    f"""
Propose {NUMBER_OF_ROUTES} different synthesis routes for the target {TARGET}.

Constraints from the user:
{CONSTRAINTS}

Validate every route before returning it, and give each route's steps in forward
order. Before returning the routes, say clearly if a route could not be fully
validated, or if a constraint could not be met.
""",
    output_type=list[sa.Route],
)

for number, route in enumerate(routes.output, start=1):
    display(Markdown(f"### Route {number}"), route)

╭───────────────────────────────────────────── Tool: load_capability ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "id": "synthesis-validation"                                                                                  │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Tool: load_capability ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "id": "analogue-search"                                                                                       │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Result: load_capability ────────────────────────────────────────────╮
│ {"instructions":"Validate a synthesis route with these tools. Report exactly what the tools return. If you edit │
│ a SMILES or template, validate the edited version; a step passes only on the version that was                   │
│ validated.\n\nProcedure:\n1. Collect every molecule in the route: building blocks, intermediates, and the       │
│ target. Check them in a single validate_smiles call.\n2. Check all reaction templates in a single               │
│ validate_reaction_smarts call.\n3. For each step, call validate_products with the template…                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Result: load_capability ────────────────────────────────────────────╮
│ {"instructions":"Use the analogue searching capability to find building blocks that fit reaction templates. Use │
│ check_building_blocks and check_templates to confirm that a route only uses building blocks and templates from  │
│ the libraries."}                                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Tool: validate_smiles ─────────────────────────────────────────────╮
│ {                                                                                                               │
│   "smiles": [                                                                                                   │
│     "O=C(C1=CC=CC(C(F)(F)F)=C1)NC2=CC(C3=CC4=C(C(C=CC=C5)=C5C=C4)N3)=CC=N2"                                     │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────── Tool: check_building_blocks ──────────────────────────────────────────╮
│ {                                                                                                               │
│   "smiles": [                                                                                                   │
│     "O=C(C1=CC=CC(C(F)(F)F)=C1)NC2=CC(C3=CC4=C(C(C=CC=C5)=C5C=C4)N3)=CC=N2"                                     │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (156 outputs left out)

### Route 2

SynLlamaFormat(reactions=[SynLlamaReaction(reaction_number=1, reaction_template='([Cl].[cH1:1][c:2][NH:3][NH2:4]).[CH3:5][C:6](=[O:7])[c:8]>>[c:1]1[c:2][nH:3][c:6](-[c:8])[c:5]1', reactants=['Cl.NNc1cccc2ccccc12', 'CC(=O)c1ccnc(N)c1'], product='Nc1cc(-c2cc3ccc4ccccc4c3[nH]2)ccn1'), SynLlamaReaction(reaction_number=2, reaction_template='[NX3;!$(N[C]=[S,O,N]);$(N[#6,S]);!$(N[#7]);H1,H2:1].[#6:4][C:5](=[O:6])[OH,O-]>>[N:1][C:5](=[O:6])[#6:4]', reactants=['Nc1cc(-c2cc3ccc4ccccc4c3[nH]2)ccn1', 'O=C(O)c1cccc(C(F)(F)F)c1'], product='O=C(C1=CC=CC(C(F)(F)F)=C1)NC2=CC(C3=CC4=C(C(C=CC=C5)=C5C=C4)N3)=CC=N2')], building_blocks=['Cl.NNc1cccc2ccccc12', 'CC(=O)c1ccnc(N)c1', 'O=C(O)c1cccc(C(F)(F)F)c1'])

### Route 3

SynLlamaFormat(reactions=[SynLlamaReaction(reaction_number=1, reaction_template='[c;H0:1][Br]>>[c;H1:1]', reactants=['Brc1cccc2c1ccc1cc[nH]c12'], product='c1ccc2c(c1)ccc1cc[nH]c12'), SynLlamaReaction(reaction_number=2, reaction_template='[#6:1][C:2](=[O:3])[F,Cl,Br].[NX3;!$(N[C,S]=[S,O,N]);$(N[#6]);!$(N[#7]);H1,H2:4]>>[#6:1][C:2](=[O:3])[N:4]', reactants=['O=C(Cl)c1cccc(C(F)(F)F)c1', 'Nc1cc(Br)ccn1'], product='O=C(Nc1cc(Br)ccn1)c1cccc(C(F)(F)F)c1'), SynLlamaReaction(reaction_number=3, reaction_template='[cH1:1]1[nH:2][c:3][c:4][c:5]1.[Br,I,Cl][c:6]>>[c:1]1(-[c:6])[nH:2][c:3][c:4][c:5]1', reactants=['c1ccc2c(c1)ccc1cc[nH]c12', 'O=C(Nc1cc(Br)ccn1)c1cccc(C(F)(F)F)c1'], product='O=C(C1=CC=CC(C(F)(F)F)=C1)NC2=CC(C3=CC4=C(C(C=CC=C5)=C5C=C4)N3)=CC=N2')], building_blocks=['Brc1cccc2c1ccc1cc[nH]c12', 'O=C(Cl)c1cccc(C(F)(F)F)c1', 'Nc1cc(Br)ccn1'])

## Step 4: Rank the routes

SynAgent now ranks the routes it just proposed. It always judges synthetic feasibility: whether every step passed validation, how many steps a route has, how reliable each reaction is in practice, and whether the building blocks are available. It then weighs your own priorities.

### Edit this

`RANKING_PRIORITIES` lists what matters most to you, in plain English, one per line. Keep the triple quotes (`"""`) at the start and end. To rank on feasibility alone, write `None` between them.

You could also ask it to *prefer routes that avoid chromatography*, *prefer routes I can run in a day*, or *prefer building blocks I can buy from one supplier*.

In [5]:
RANKING_PRIORITIES = """
Prefer routes with the fewest steps.
Prefer common, inexpensive building blocks.
Penalise steps that need careful handling or specialised equipment.
"""

In [6]:
ranking = await chat.ask(f"""
Rank the routes you just proposed, best first.

Judge synthetic feasibility on:
- Validation: a route with failed or unresolved steps ranks below every verified route.
- Length: fewer steps is better, and a convergent route beats a linear one of the same length.
- Reliability: how robust each reaction is in practice, and how likely side reactions,
  competing reactive groups, or protecting groups are to cause trouble.
- Building blocks: whether each one is in the library and how common it is.

Then weigh these priorities from the user:
{RANKING_PRIORITIES}

Present the ranking as a numbered list, best first. For each route give one line on
why it sits there, its number of steps, and its main risk. Where a route ranks lower
only because of the user's priorities rather than its feasibility, say so. Finish with
one short paragraph recommending a route and what to check before trying it at the bench.
""")

Route Ranking                                                                                                      

 1 Route 1 (2 steps)                                                                                               
    • Why it sits here: Shortest route (2 steps), uses inexpensive and readily accessible starting materials, and  
      installs the amide bond first to avoid any chemoselectivity issues with the indole nitrogen.                 
    • Length: 2 steps.                                                                                             
    • Main risk: Fischer indole cyclization of electron-deficient heteroaryl methyl ketones can require elevated   
      temperatures or strong Brønsted/Lewis acids (e.g., $\text{ZnCl}_2$, $\text{AcOH}$, or polyphosphoric acid)   
      which may risk partial hydrolysis of the benzamide.                                                          
 2 Route 2 (2 steps)                                                                                               
    • Why it sits here: Equally short (2 steps) and utilizes simple starting materials, but ranks below Route 1 due
      to the risk of competing acylation at the unprotected indole nitrogen.                                       
    • Length: 2 steps.                                                                                             
    • Main risk: In the final amide coupling, the intermediate contains both an exocyclic 2-aminopyridine          
      $\text{NH}_2$ and a nucleophilic indole $\text{NH}$, risking diacylation or off-target $N$-acylation.        
 3 Route 3 (3 steps)                                                                                               
    • Why it sits here: Ranks lowest primarily due to user priorities: it is longer (3 steps), requires            
      pressurized/pyrophoric hydrogenation conditions ($\text{H}_2/\text{Pd-C}$), and uses a specialized, expensive
      starting material (9-bromo-1$H$-benzo[$g$]indole).                                                           
    • Length: 3 steps.                                                                                             
    • Main risk: Catalytic hydrodebromination requires flammable hydrogen gas handling, and the subsequent         
      intermolecular direct $\text{C–H}$ arylation of 1$H$-benzo[$g$]indole at C-2 can suffer from poor            
      regioselectivity ($\text{C-2 vs. C-3}$ arylation) and catalyst inhibition by the pyridylbenzamide ligand.    

-------------------------------------------------------------------------------------------------------------------

Recommendation                                                                                                     

Route 1 is strongly recommended for bench execution because it avoids high-pressure hydrogenation, utilizes cheap  
and robust commercial building blocks, and sequences the reactions so that the amide is assembled prior to         
heterocycle condensation. Before attempting this route in the laboratory, verify the stability of the              
3-(trifluoromethyl)benzamide moiety under standard Fischer indole cyclization conditions (e.g., refluxing glacial  
acetic acid or ethanolic $\text{HCl}$); if mild acid-catalyzed condensation proves slow due to the                 
electron-withdrawing pyridyl ring, pre-forming the hydrazone in neutral ethanol followed by Lewis acid activation  
($\text{ZnCl}_2$ or $\text{BF}_3\cdot\text{OEt}_2$) will minimize amide cleavage.

## Step 5: Save the results

This cell saves the target, your settings, the routes, and the ranking to a Markdown file named `synagent-results.md`. The file goes in the `notebook-examples` folder, and the cell prints its full path. Running the cell again overwrites the file, so change `OUTPUT_FILE` first if you want to keep an earlier set of results.

In [7]:
from datetime import datetime
from pathlib import Path

OUTPUT_FILE = "synagent-results.md"

routes_text = "\n\n".join(
    f"### Route {number}\n\n```json\n{route.model_dump_json(indent=2)}\n```"
    for number, route in enumerate(routes.output, start=1)
)

Path(OUTPUT_FILE).write_text(
    f"# SynAgent results\n\n"
    f"Target: `{TARGET}`  \n"
    f"Model: {MODEL}  \n"
    f"Date: {datetime.now():%Y-%m-%d %H:%M}\n\n"
    f"## Constraints\n{CONSTRAINTS}\n\n"
    f"## Routes\n\n{routes_text}\n\n"
    f"## Ranking priorities\n{RANKING_PRIORITIES}\n\n"
    f"## Ranking\n\n{ranking.text}\n"
)

print(f"✅ Saved to {Path(OUTPUT_FILE).resolve()}")

✅ Saved to /Users/lkim/Projects/agents/SynAgent/notebook-examples/synagent-results.md


## Step 6: Ask follow-up questions (optional)

SynAgent remembers this whole conversation, so you can ask more about the routes it proposed. Change the question in `FOLLOW_UP` and run the cell, as many times as you like.

Some questions to try:

- *Why did you rank the top route above the others?*
- *Are there alternative building blocks for step 2 of Route 1?*
- *Propose one more route that avoids the piperazine alkylation.*
- *What could go wrong with the amide coupling in Route 2?*

Running the Step 3 cell again starts a new conversation.

In [8]:
FOLLOW_UP = "Why did you rank the top route above the others?"

await chat.ask(FOLLOW_UP)

Route 1 was ranked above the others because it achieves the best balance of chemoselectivity, operational          
simplicity, and user priorities:                                                                                   

 1 Superior Chemoselectivity over Route 2 (Both are 2 steps):                                                      
    • In Route 2, assembling the 1$H$-benzo[$g$]indole core first leaves an intermediate with two nucleophilic     
      nitrogen centers: the exocyclic 2-aminopyridine $\text{NH}_2$ and the indole ring $\text{NH}$. Subsequent    
      acylation presents a significant risk of competitive $N$-acylation at the indole nitrogen or unwanted        
      di-acylation, requiring either careful stoichiometry or an extra protection/deprotection sequence.           
    • In Route 1, acylation is performed first on 4-acetylpyridin-2-amine where only a single nucleophilic primary 
      amine is present. The amide bond is established cleanly and quantitatively without competing nucleophiles,   
      after which the ketone cleanly undergoes cyclization with 1-naphthylhydrazine.                               
 2 Fewer Steps and Lower Risk than Route 3 (2 steps vs. 3 steps):                                                  
    • Step Count: Route 1 directly satisfies the user's preference for the fewest steps (2 steps vs. 3 steps).     
    • No Hazardous Equipment: Route 3 requires catalytic hydrodebromination with $\text{H}_2$ gas and pyrophoric   
      $\text{Pd/C}$, conflicting with the user's priority to penalize routes requiring careful handling or         
      specialized pressure equipment.                                                                              
    • Regioselectivity and Catalysis: Route 3 relies on a late-stage intermolecular direct $\text{C–H}$ arylation  
      of 1$H$-benzo[$g$]indole. Without a directing group, direct arylation of indoles frequently suffers from     
      $\text{C-2 vs. C-3}$ selectivity issues and catalyst poisoning by coordinating pyridyl groups. Route 1’s     
      Fischer indole cyclization, by contrast, is regiochemically unambiguous.                                     
 3 Cost and Availability of Starting Materials:                                                                    
    • Route 1 uses simple, commodity-grade starting materials (4-acetylpyridin-2-amine, 3-(trifluoromethyl)benzoyl 
      chloride, and 1-naphthylhydrazine hydrochloride).                                                            
    • Route 3 requires 9-bromo-1$H$-benzo[$g$]indole, a rare, expensive polycyclic intermediate that must          
      immediately be debrominated in the first step.

## If something goes wrong

### "No Gemini API key found"

The `.env` file is missing, in the wrong folder, or has a typo. It belongs in the `SynAgent` folder next to `README.md`, and it should contain one line: `GOOGLE_API_KEY=your-key`. After you fix it, choose **Kernel → Restart Kernel** and run the cells again from the top.

### A message about a quota, a rate limit, or `429`

You've reached the free Gemini tier's limits. Wait a few minutes and run the cell again, or check your usage at https://aistudio.google.com/api-keys.

### "ModuleNotFoundError: No module named 'synagent'"

Jupyter was started outside the project's environment. Close it, then start it again from inside the `SynAgent` folder with `uv run --extra jupyter jupyter lab notebook-examples/generate-and-rank.ipynb`.

If you installed without uv, activate the environment before starting Jupyter, as shown in "Opening the notebook". The terminal prompt should start with `(.venv)`.

### "No module named 'FPSim2.FPSim2Reactions'"

pip installed the standard FPSim2 instead of the version SynAgent needs. This happens when `pip install ".[jupyter]"` runs before the FPSim2 line. Activate the environment and run:

```sh
pip install --force-reinstall --no-deps "fpsim2 @ git+https://github.com/lukasmki/FPSim2.git"
```

### "running scripts is disabled on this system" (Windows)

PowerShell blocks the script that activates the environment. Allow scripts for your user account once, then run `.venv\Scripts\Activate.ps1` again:

```powershell
Set-ExecutionPolicy -Scope CurrentUser RemoteSigned
```

### A cell seems stuck

Generating and ranking take a few minutes, and new tool boxes keep appearing while SynAgent works. If nothing new appears for a long time, choose **Kernel → Interrupt Kernel** and run the cell again.

### SynAgent can't validate a route

When SynAgent can't ground a step in its libraries, it reports the step as unresolved. Try a different target, loosen your constraints, or use the follow-up cell to ask why a step failed.

### Prices and availability

SynAgent looks up prices on Chemspace, which needs its own API key. Without one, SynAgent tells you that pricing is unavailable. To turn it on, add a second line to `.env`: `CHEMSPACE_API_KEY=your-key`.

## Where to go next

- `notebook-examples/overview.ipynb` covers the rest of the Python interface, including tool call inspection, routes built by hand, and the validation workflows.
- The documentation explains the command line tool, the other configurations, and how to change what SynAgent can do. Build it with `make -C docs html`.